# AI for formulation {#sec-ai-formulate}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/book-ai/notebooks/c19-ai-formulate.ipynb)

Chapter 3 ended with a dynamic hypothesis: an edge list of causal links and the loops they make. Chapters 5 to 9 turned those loops into equations by hand. Formulation takes most of a modeller's hours, and most errors are born there: a function the software lacks, units that do not balance, a decision rule that reads information nobody has. This chapter asks how language models and machine learning can share that work, and how code checks every piece they produce.

```
            Perceived wait <-------------------- Waiting time
                  |       GPs perceive the wait        ^
                  |       over 6 months (B1)           |
                  v                                    |
 Need ------> Referrals ------> [ Waiting list ] ---> Treatments
                                       |                  ^
                                       | monthly report   |
                                       v                  |
                                 [ Reported list ]        |
                                       |                  |
                                       v                  |
                                    Pressure --> Overtime --> Capacity
                                         working harder (B3)
```

*The structure formulated in this chapter: two loops of chapter 3's hypothesis, Discouraged referrals (B1) and Working harder (B3), with the stocks that perception and reporting add. A language model drafts it, an agent repairs it, a reasoning model reviews it, and code checks each step.*

The chapter follows one formulation from draft to documented model. A **generative** model drafts a loopbricks spec from the hypothesis, and loading it is the first check (section 1). An **agent** repairs the draft with tools we write: a load check, a units checker, a link checker and extreme runs (section 2). A **reasoning** model reviews the decision rules, and code checks its review (section 3). Machine learning recovers an equation from data (section 4). A model writes the documentation, and code checks it against the spec (section 5). A failure gallery collects the ways this stage goes wrong.

**The replies are recorded.** Every model reply in this chapter is stored in `data/ai/c19.json` and replayed, as @sec-ai-partners explains. The replies were written for the book with claude-opus-5-5; none came from a live call, and each record says so in its `how` field. We wrote the drafts to contain the kinds of mistake language models make in formulation, so the checks have something to catch. The text says where a reply holds a fault we planted. To ask a live model instead, run `pip install anthropic`, set `ANTHROPIC_API_KEY` and `LB_AI_LIVE=1`, and change a prompt: the new reply is recorded. A live model will draft differently. The checks are what carry over.

**Who it is for.** Modellers who want help from AI in formulation without losing control of the model. **Who can skip.** Readers who know sparse regression can skip section 4. Readers who will never use a language model should still read sections 2 and 3: the units, link and information checkers work just as well on equations written by hand. **Prerequisites.** @sec-ai-partners for language models, reasoning models, agents and replay files; the edge list of @sec-story-to-diagram; perceived waits from @sec-delays; decision rules from @sec-decisions; the spec format of @sec-package. **Time.** About three hours.

**Learning outcomes.** By the end of this chapter you can:

1. **draft** a model spec from a dynamic hypothesis with a language model, and **check** that it loads (section 1, Try it 1);
2. **build** a units checker and a link checker, **run** an agent that repairs a spec with them, and **verify** its result independently (section 2, Try it 2);
3. **critique** decision rules with a reasoning model and **verify** each criticism against the equations (section 3, Try it 3);
4. **recover** a known molecule's equation from noisy data by sparse regression (section 4, Try it 4);
5. **generate** model documentation and **check** its names and numbers against the spec (section 5, Try it 5);
6. **diagnose** the ways AI goes wrong in formulation, each with a check that catches it (failure gallery, Try it 6).

In [ ]:
# Run this cell first. In Colab it installs loopbricks (about ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/"
                           "loopbricks.git@book-ai#subdirectory=python"])
    import loopbricks as lb
import copy
import json
import os
import re
from collections import Counter
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from loopbricks import ai

SEED = 19
rng = np.random.default_rng(SEED)

GITHUB = ("https://raw.githubusercontent.com/waelrash1/loopbricks/main/"
          "notebooks/")
DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") \
    else GITHUB + "data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])
# Every model reply of this chapter is recorded in this file (see chapter 2)
REPLAY = "data/ai/c19.json" if os.path.exists("data/ai/c19.json") \
    else GITHUB + "data/ai/c19.json"

# Okabe-Ito colours, each with its own line style, to read in greyscale.
OKABE_ITO = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#D55E00",
             "#0072B2", "#CC79A7"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1)),
          (0, (1, 3))]
plt.rcParams["axes.prop_cycle"] = cycler(color=OKABE_ITO) + \
    cycler(linestyle=STYLES)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__, "| model", ai.MODEL, "| replies from",
      REPLAY)

## 1. From hypothesis to spec: a generative draft

**Stage task:** turn part of the dynamic hypothesis into equations. **Kind of AI:** generative. **Check:** the spec loads with `lb.Model.from_spec`. **What stays human:** which part of the hypothesis to formulate, the numbers in the prompt, and whether to accept the draft.

We formulate two loops of chapter 3's hypothesis. B1, Discouraged referrals, runs from the list to the waiting time to referrals: GPs refer fewer patients when they believe the wait is long. B3, Working harder, runs from the list to pressure, overtime and capacity. Together they use nine of the edge list's 22 links. The units come from chapter 3's variable table, with one simplification: capacity is in patients a month (one appointment per patient), so severity can wait.

In [ ]:
HYPOTHESIS = pd.DataFrame([
    # cause, effect, polarity, who said so (chapter 3's edge list)
    ("Need in the population", "Referrals", "+", "Patient advocate"),
    ("Referrals", "Waiting list", "+", "inflow"),
    ("Treatments", "Waiting list", "-", "outflow"),
    ("Waiting list", "Waiting time", "+", "definition"),
    ("Waiting time", "Referrals", "-", "GP"),
    ("Capacity", "Treatments", "+", "Consultant"),
    ("Waiting list", "Pressure", "+", "Manager"),
    ("Pressure", "Overtime", "+", "Consultant"),
    ("Overtime", "Capacity", "+", "Consultant"),
], columns=["cause", "effect", "sign", "source"])
VARIABLES = pd.DataFrame([
    ("Need in the population", "auxiliary", "patients/month"),
    ("Referrals", "flow", "patients/month"),
    ("Waiting list", "stock", "patients"),
    ("Waiting time", "auxiliary", "months"),
    ("Treatments", "flow", "patients/month"),
    ("Capacity", "auxiliary", "patients/month"),   # one appointment each
    ("Pressure", "auxiliary", "dimensionless"),
    ("Overtime", "auxiliary", "extra clinics/month"),
], columns=["variable", "kind", "units"])
print(HYPOTHESIS.to_string(index=False))

In [ ]:
def stock(i, name, init, units):
    """A stock for a model spec, held at or above zero."""
    return {"id": i, "name": name, "init": str(init), "units": units,
            "nonNegative": True}


def flow(i, name, eq, units, src=None, dst=None):
    """A flow for a model spec, from stock src to stock dst (by id)."""
    f = {"id": i, "name": name, "eq": eq, "units": units}
    if src:
        f["from"] = src
    if dst:
        f["to"] = dst
    return f


def var(i, name, eq, units):
    """A variable for a model spec: a constant or an equation."""
    return {"id": i, "name": name, "eq": str(eq), "units": units}


EXAMPLE = {"name": "Clinic list", "timeUnit": "month", "T": 36, "dt": 0.25,
           "stocks": [stock("wl", "Waiting list", 20000, "patients")],
           "flows": [flow("in", "Referrals", "Base_referrals",
                          "patients/month", dst="wl"),
                     flow("out", "Treatments",
                          "MIN(Capacity, Waiting_list / Minimum_time)",
                          "patients/month", src="wl")],
           "vars": [var("br", "Base referrals", 4000, "patients/month"),
                    var("cap", "Capacity", 3900, "patients/month"),
                    var("mt", "Minimum time", 0.5, "month")]}

SYSTEM = ("You are an experienced system dynamics modeller. You write "
          "models as loopbricks specs: JSON objects that the Python "
          "package loopbricks loads with lb.Model.from_spec.")
PROMPT = f"""Formulate part of a dynamic hypothesis as a loopbricks spec.

Links of the hypothesis (cause, effect, polarity, who said so):
{HYPOTHESIS.to_string(index=False)}

Variables, with the units the interviews suggest:
{VARIABLES.to_string(index=False)}

The two loops are B1 Discouraged referrals (GPs refer fewer patients
when they believe the wait is long, and they perceive the wait with a
delay) and B3 Working harder (pressure from the list brings overtime,
which adds capacity).

Numbers: the list starts at 32,000 patients. Need is 4,000
patients/month and rises by 400 patients/month in month 12. Normal
capacity is 4,000 patients/month and the minimum treatment time 0.5
month. The reference wait is 8 months, GPs perceive the wait over 6
months, and the referral sensitivity is 0.3. Time is in months: run
60 months with dt 0.125.

The spec format, by example:
{json.dumps(EXAMPLE, indent=1)}

Every element has an id, a name and units; a flow has "from" and/or
"to" (stock ids). Equations name other elements with underscores for
spaces, in a syntax close to Vensim's. Reply with the JSON spec only."""
print(len(PROMPT.split()), "words in the prompt")

In [ ]:
def spec_from(reply):
    """The JSON object in a reply, from its first { to its last }."""
    return json.loads(reply[reply.index("{"): reply.rindex("}") + 1])


def elements(spec):
    """Every stock, flow and variable of a spec, each with its kind."""
    return ([dict(e, kind="stock") for e in spec.get("stocks", [])]
            + [dict(e, kind="flow") for e in spec.get("flows", [])]
            + [dict(e, kind="var") for e in spec.get("vars", [])])


def equation(e):
    """The equation of an element: a stock's initial value, else its eq."""
    return str(e["init"] if e["kind"] == "stock" else e["eq"])


def listing(spec):
    """A spec as text: one line per element, with its units."""
    rows = []
    for e in elements(spec):
        ends = "".join(f", {k} {e[k]}" for k in ("from", "to") if e.get(k))
        rows.append(f"{e['kind']} {e['name']} (id {e['id']}{ends}) = "
                    f"{equation(e)} [{e.get('units', '')}]")
    return "\n".join(rows)


def load(spec):
    """The first check: the loaded model, or the error message as text."""
    try:
        return lb.Model.from_spec(spec)
    except lb.LoopBricksError as e:
        return f"error: {e}"


reply1 = ai.ask(PROMPT, system=SYSTEM, replay=REPLAY)
draft1 = spec_from(reply1)
print(listing(draft1))
print()
print(load(draft1))

The prompt gives the links, the units, the two loops in words, the numbers and the spec format by example. It says the syntax is "close to Vensim's" and does not list the functions. That is how people usually ask, and it invites the first mistake.

The draft has 16 elements and reads well. Referrals use the power form of chapter 6, the list starts at 32,000 patients and the overtime rule is capped. It does not load: `Perceived wait` calls `SMOOTH`, a Vensim function our equation language does not have. A **hallucinated function** is a fluent, plausible name for something the software lacks; it is one kind of the confident errors that language models make [@ji2023hallucination]. Notice also two numbers the prompt never gave: a maximum overtime of 0.2 and a response of 0.5 inside the overtime rule. The model invented them. Section 3 comes back to them.

`from_spec` stops at the first error. The function below compiles every equation on its own, so it lists all of them at once. Then we send the error back, the cheapest repair there is.

In [ ]:
def equation_errors(spec):
    """Every equation that does not compile or names an unknown element."""
    known = {lb.name_key(e["name"]) for e in elements(spec)}
    out = []
    for e in elements(spec):
        try:
            refs = lb.compile_expr(equation(e)).refs
        except lb.LoopBricksError as err:
            out.append(f"{e['name']}: {err}")
            continue
        out += [f'{e["name"]}: unknown name "{r}"' for r in refs
                if r not in known]
    return out


print(equation_errors(draft1))


def fix_request(reply, error):
    """The conversation so far, plus the error, asking for a fixed spec."""
    return [{"role": "user", "content": PROMPT},
            {"role": "assistant", "content": reply},
            {"role": "user", "content": "Loading the spec gave this "
             f"{error}\nFix the spec and reply with the whole JSON spec "
             "only."}]


reply2 = ai.complete(fix_request(reply1, load(draft1)), system=SYSTEM,
                     replay=REPLAY)
draft2 = spec_from(reply2)
print(equation_errors(draft2) or "every equation compiles")
print(load(draft2))

Here `SMOOTH` is the only error. Draft 2 replaces it with a stock and its updating flow, exactly as the Smooth molecule (`smooth`) of chapter 6 builds a perception (after Hines, *Molecules of Structure* [@hines_molecules]). Every equation now compiles, yet the spec still does not load. Five elements depend on each other in a circle with no stock between them: Treatments, Capacity, Overtime, Pressure, Waiting time and back to Treatments.

Such a circle is a **simultaneous equation**, often called an **algebraic loop**: to compute Treatments this step you need Waiting time, which needs Treatments. In a stock-and-flow model every feedback loop must pass through a stock, because feedback takes time [@sterman2000]. Chapter 3's Check it tested this rule on the diagram; the draft broke it in the equations, when `Pressure` read `Waiting_time`. Back it goes.

In [ ]:
messages3 = fix_request(reply1, load(draft1)) + [
    {"role": "assistant", "content": reply2},
    {"role": "user", "content": f"Loading the new spec gave this "
     f"{load(draft2)}\nFix the spec and reply with the whole JSON spec "
     "only."}]
reply3 = ai.complete(messages3, system=SYSTEM, replay=REPLAY)
draft3 = spec_from(reply3)
model3 = load(draft3)
print(model3)
print()
print(model3.equations())

Draft 3 loads: two stocks, three flows and twelve variables. Loading checks syntax, names, the ends of each flow and the order of computation. It does not check meaning: whether the units balance, whether the whole hypothesis is there, or whether the model behaves.

Look at how the model broke the circle. `Pressure` now reads `Perceived_wait`, a stock, so the spec loads. But the managers' pressure now comes from the GPs' belief about the wait, and the hypothesis says it comes from the list. The spec loads and is wrong. Section 2's checks catch it.

**Try it 1.**

- *Check.* How many elements does draft 3 have, and which element is new since draft 1? What happened to `Perceived wait`?
- *Apply.* Copy draft 1 and misspell a name in `Pressure` as well. Compare what `load` and `equation_errors` report.
- *Extend.* Rewrite the prompt so it lists the functions of the equation language. Ask for a draft offline and read what happens. Then explain why the load check must stay even with the better prompt.

<details><summary>Worked answer</summary>

*Check.* Seventeen elements. The only new name is `Updating perceived wait`; `Perceived wait` kept its name but changed from a variable with `SMOOTH` into a stock with that flow as its inflow.

*Apply.* `load` reports only the first error, the unknown function `SMOOTH`. `equation_errors` reports both: `SMOOTH` and the unknown name `waiting_tme`. One repair round then fixes both.

*Extend.* The changed prompt has a new replay key, so offline it raises `ai.MissingReply`: no reply was recorded for it. With a key and `LB_AI_LIVE=1`, a live model answers and the reply is recorded. Listing the functions makes a hallucinated function less likely, not impossible, and it does nothing for the other errors, such as the algebraic loop. The prompt lowers the error rate; the check catches what gets through.

</details>

In [ ]:
names1 = {e["name"] for e in elements(draft1)}
print("draft 3:", len(elements(draft3)), "elements; new since draft 1:",
      sorted({e["name"] for e in elements(draft3)} - names1))
two = copy.deepcopy(draft1)
for v in two["vars"]:
    if v["name"] == "Pressure":
        v["eq"] = "Waiting_tme / Reference_wait"     # a misspelt name too
print("from_spec:", load(two))
print("equation_errors:", equation_errors(two))
FUNCTIONS = "MIN, MAX, ABS, EXP, LN, SQRT, IF, STEP, PULSE, RAMP, TABLE, TIME"
PROMPT2 = PROMPT.replace("in a syntax close to Vensim's",
                         "using only these functions: " + FUNCTIONS)
try:
    ai.ask(PROMPT2, system=SYSTEM, replay=REPLAY)
except ai.MissingReply as e:
    print("offline:", str(e)[:40], "...")

## 2. An agent that writes, checks and runs

**Stage task:** repair a draft until it loads, balances its units, contains the hypothesis and behaves. **Kind of AI:** an agent that calls tools in a loop [@yao2022react]. **Check:** the final spec is checked again by our own code, independently of what the agent says, and every number in its answer must come from a tool. **What stays human:** the tools, the tests the agent must pass, and every change to declared units.

An agent needs tools that tell it the truth. Loading is one. The second is a **units checker**. A model is **dimensionally consistent** when every term of a sum has the same units and every flow has its stock's units per unit of time. Chapter 7's Check it did this by eye; here we write it in code. The first step reads a units text, such as `patients/clinician/month`, as a dictionary of base units and powers. Plural endings are dropped, so `months` and `month` are the same unit, and `fraction` and `dimensionless` have no units at all.

In [ ]:
TIME_WORDS = {"", "1", "dimensionless", "fraction", "ratio"}


def parse_units(text):
    """Units such as "patients/clinician/month" as {base unit: power}."""
    out = {}
    for k, part in enumerate(str(text).lower().split("/")):
        for word in part.split("*"):
            w = "_".join(word.split())            # "extra clinics"
            if w in TIME_WORDS:
                continue
            if w.endswith("s") and not w.endswith("ss"):
                w = w[:-1]                        # patients -> patient
            out[w] = out.get(w, 0) + (1 if k == 0 else -1)
    return {b: p for b, p in out.items() if p}


def show_units(u):
    """{base unit: power} as text, such as "patient/month"."""
    if not u:
        return "dimensionless"
    up = [b if p == 1 else f"{b}^{p}" for b, p in sorted(u.items()) if p > 0]
    down = [b if p == -1 else f"{b}^{-p}" for b, p in sorted(u.items())
            if p < 0]
    return "*".join(up or ["1"]) + "".join("/" + b for b in down)


def times(a, b, sign=1):
    """The units of a * b (sign 1) or a / b (sign -1)."""
    out = dict(a)
    for base, p in b.items():
        out[base] = out.get(base, 0) + sign * p
    return {k: p for k, p in out.items() if p}


print(show_units(parse_units("patients/clinician/month")), "|",
      show_units(parse_units("months/month")), "|",
      show_units(times(parse_units("patients"), parse_units("months"), -1)))

In [ ]:
def infer(node, units, problems, where):
    """The units of a parsed equation, or None for a pure number.

    A pure number takes the units of whatever it is added to, so
    "Waiting_list + 100" is in patients. Problems are appended to the list.
    """
    kind = node[0]
    if kind == "num":
        return None
    if kind == "name":
        return units.get(node[1])
    if kind in ("time", "dt"):
        return units["TIME"]
    if kind == "neg":
        return infer(node[1], units, problems, where)
    if kind == "bin":
        op = node[1]
        a = infer(node[2], units, problems, where)
        b = infer(node[3], units, problems, where)
        if op in ("*", "/"):
            if a is None and b is None:
                return None
            return times(a or {}, b or {}, 1 if op == "*" else -1)
        if op == "^":
            if node[3][0] == "num" and a:          # x ^ 2: powers scale
                return {k: p * node[3][1] for k, p in a.items()}
            if a or b:
                problems.append(f"{where}: a power needs a dimensionless "
                                f"base and exponent, not {show_units(a)} "
                                f"^ {show_units(b)}")
            return None if a is None else {}
        if op in ("+", "-"):
            return same([a, b], problems, where, "adds")
        same([a, b], problems, where, "compares")    # gives 1 or 0
        return None
    name, args = node[1], [infer(x, units, problems, where) for x in node[2]]
    if name in ("MIN", "MAX", "ABS"):
        return same(args, problems, where, "mixes in " + name)
    if name == "IF":
        return same(args[1:], problems, where, "mixes in IF")
    if name in ("STEP", "PULSE"):
        return args[0]
    if name == "RAMP":
        return times(args[0] or {}, units["TIME"])
    if name in ("EXP", "LN", "SQRT"):
        if args[0]:
            problems.append(f"{where}: {name} of {show_units(args[0])}")
        return None
    return None              # TABLE: a pure number; its input is not checked


def same(us, problems, where, verb):
    """The shared units of several terms; a problem if they differ."""
    known = [u for u in us if u is not None]
    for u in known[1:]:
        if u != known[0]:
            problems.append(f"{where}: {verb} {show_units(known[0])} and "
                            f"{show_units(u)}")
            break
    return known[0] if known else None


def unit_problems(spec):
    """Every place where the units of a spec do not balance."""
    els = elements(spec)
    units = {lb.name_key(e["name"]): parse_units(e["units"])
             for e in els if "units" in e}
    units["TIME"] = parse_units(spec.get("timeUnit", ""))
    stocks = {s["id"]: s for s in spec.get("stocks", [])}
    problems = []
    for e in els:
        try:
            tree = lb.compile_expr(equation(e)).ast
        except lb.LoopBricksError:
            continue                    # load() reports equations that fail
        got = infer(tree, units, problems, e["name"])
        want = units.get(lb.name_key(e["name"]))
        if got is not None and want is not None and got != want:
            problems.append(f"{e['name']}: the equation gives "
                            f"{show_units(got)}, the declared units are "
                            f"{show_units(want)}")
        for end in ("from", "to"):
            s = stocks.get(e.get(end)) if e["kind"] == "flow" else None
            need = s and times(parse_units(s["units"]), units["TIME"], -1)
            if s and need != want:
                problems.append(f"{e['name']}: a flow of {s['name']} must "
                                f"be in {s['units']} per "
                                f"{spec.get('timeUnit')}, which is "
                                f"{show_units(need)}, not "
                                f"{show_units(want)}")
    return problems


for p in unit_problems(draft3):
    print(p)

The checker walks the parse tree of each equation, the same tree the package compiles (`lb.compile_expr(eq).ast`). Names take their declared units, and `*` and `/` combine units. `+`, `-`, `MIN` and `MAX` need the same units on every side. A power needs a dimensionless base and exponent, unless the exponent is a plain number. A plain number takes the units of what it is added to, so `Waiting_list + 100` is in patients. A `TABLE` gives a pure number and its input is not checked, as in many tools; the failure gallery shows what that leniency costs.

Draft 3 has four problems in three elements. `Updating perceived wait` is declared in months. But a gap in months divided by a time in months is dimensionless, and so is the months per month a flow into its stock must have. `Overtime` gives a fraction but is declared in extra clinics a month, the units of chapter 3's table. `Capacity` adds that to patients a month. The checker cannot say which side is wrong, the equation or the declared units. That takes judgement.

The third check compares the equations with the hypothesis. An element reads the names in its equation, and a stock reads its flows. Elements added in formulation, such as the perceived wait, are looked through, so the link from waiting time to referrals still counts when it passes through the GPs' perception.

In [ ]:
def link_problems(spec, hyp):
    """Links of the hypothesis missing from the equations, and links in the
    equations between hypothesis variables that the hypothesis lacks.

    An element "reads" the names in its equation; a stock reads its flows.
    Elements added in formulation (such as a perceived wait) are looked
    through, so a link may pass through them.
    """
    els = {lb.name_key(e["name"]): e for e in elements(spec)}
    shown = {lb.name_key(n): n for n in set(hyp.cause) | set(hyp.effect)}

    def reads(e):
        if e["kind"] == "stock":
            return {lb.name_key(f["name"]) for f in spec.get("flows", [])
                    if e["id"] in (f.get("from"), f.get("to"))}
        return set(lb.compile_expr(equation(e)).refs)

    def sources(key):
        seen, todo, found = set(), list(reads(els[key])), set()
        while todo:
            k = todo.pop()
            if k in seen or k not in els:
                continue
            seen.add(k)
            if k in shown:
                found.add(k)            # a hypothesis variable: stop here
            else:
                todo += reads(els[k])   # added in formulation: look through
        return found

    missing, extra = [], []
    for key, name in sorted(shown.items()):
        if key not in els:
            missing.append(f"{name} is not in the spec")
            continue
        want = {lb.name_key(c) for c in hyp.cause[hyp.effect == name]}
        got = sources(key)
        missing += [f"{shown[c]} -> {name}" for c in sorted(want - got)]
        extra += [f"{shown[c]} -> {name}" for c in sorted(got - want)]
    return missing, extra


missing, extra = link_problems(draft3, HYPOTHESIS)
print("missing:", missing)
print("not in the hypothesis:", extra)

One link of the hypothesis is missing: Waiting list to Pressure. Three links in the equations are not in the hypothesis. Waiting time to Pressure is the draft's mistake. Waiting list to Treatments is the minimum treatment time, through which the list limits its own outflow. Treatments to Waiting time is Little's law at the current pace (chapter 6). Both are honest formulation choices that the diagram leaves out. Whether the diagram should show them is for a person to decide.

Now the tools. `make_tools` gives the agent a copy of the spec and eight functions: three checks (`load_spec`, `check_units`, `check_links`), two runs (`simulate`, `extreme`) and three edits (`set_equation`, `set_units`, `add_constant`). The agent sees each tool's name, arguments and docstring. Every number it sees comes from a tool: the runs return a summary computed by `behaviour`, not a plot it must read.

In [ ]:
def behaviour(model, **settings):
    """Summary numbers of a 60-month run of the waiting-list model."""
    d = model.run(warn_clips=False, **settings)
    lst = d["Waiting list"]
    return {"list at month 0": round(float(lst.iloc[0])),
            "list at month 12": round(float(lst.loc[12.0])),
            "highest list": round(float(lst.max())),
            "month of highest list": float(lst.idxmax()),
            "list at month 60": round(float(lst.iloc[-1])),
            "lowest value of each stock": {
                s["name"]: round(float(d[s["name"]].min()), 2)
                for s in model.stocks},
            "stocks held up by a floor": sorted(d.attrs["clipped"])}


def make_tools(spec, hyp=HYPOTHESIS):
    """Tools that let an agent read, change, check and run a copy of spec."""
    work = copy.deepcopy(spec)

    def find(name):
        for e in work["stocks"] + work["flows"] + work["vars"]:
            if lb.name_key(e["name"]) == lb.name_key(name):
                return e
        raise KeyError(f"no element called {name!r}")

    def load_spec():
        """Load the spec; returns the first error, or what was loaded."""
        m = load(work)
        return m if isinstance(m, str) else (
            f"loaded: {len(m.stocks)} stocks, {len(m.flows)} flows, "
            f"{len(m.vars)} variables")

    def check_units():
        """Check that each equation gives its declared units and that each
        flow is in its stock's units per month."""
        return unit_problems(work) or "all units balance"

    def check_links():
        """Check that the equations contain every link of the hypothesis,
        and list links between its variables that it does not have."""
        missing, extra = link_problems(work, hyp)
        return {"missing": missing, "not in the hypothesis": extra}

    def simulate():
        """Run 60 months (need rises by 400 patients/month in month 12)
        and return summary numbers."""
        m = load(work)
        return m if isinstance(m, str) else behaviour(m)

    def extreme(name: str, value: float):
        """Run 60 months with one constant set to an extreme value."""
        m = load(work)
        if isinstance(m, str):
            return m
        try:
            return behaviour(m, **{lb.name_key(name): value})
        except lb.LoopBricksError as e:
            return f"error: {e}"

    def set_equation(name: str, eq: str):
        """Replace the equation of a flow or variable."""
        find(name)["eq"] = eq
        return f"{find(name)['name']} = {eq}"

    def set_units(name: str, units: str):
        """Replace the declared units of an element."""
        find(name)["units"] = units
        return f"{find(name)['name']} is now in {units}"

    def add_constant(name: str, value: float, units: str):
        """Add a constant: a variable whose equation is a number."""
        work["vars"].append(var(f"c{len(work['vars'])}", name, value, units))
        return f"added {name} = {value} [{units}]"

    tools = {f.__name__: f for f in (load_spec, check_units, check_links,
                                     simulate, extreme, set_equation,
                                     set_units, add_constant)}
    return tools, work


tools, work = make_tools(draft3)
print(sorted(tools))

In [ ]:
AGENT_SYSTEM = ("You repair system dynamics model specs with tools. When "
                "an equation is wrong, fix the equation; change declared "
                "units only when the equation is right and its units were "
                "written down wrongly. Change as little as you can.")
TASK = f"""This draft spec formulates part of the waiting-list hypothesis:

{listing(draft3)}

Make it load, make its units balance and make it contain every link of
the hypothesis. Then check that it behaves: it should start in balance,
keep every stock at or above zero, and survive the extreme runs
normal_capacity = 0 and need_increase = -4000. In your answer, list each
change you made and why, and anything a modeller should decide."""
steps = ai.agent(TASK, tools, system=AGENT_SYSTEM, replay=REPLAY,
                 max_steps=25)
ai.show(steps, width=110)

The agent took 19 steps; read them as a log of what a careful modeller would do. It fixed the declared units of `Updating perceived wait` and `Overtime`, because their equations were right. It fixed the equation of `Capacity`, because a fraction cannot be added to patients a month. The link check showed the missing link, so it made `Pressure` read the list. That named a constant that did not exist (step 9), so it added `Target list`, set to the starting 32,000 patients so the model starts in balance. Then it ran the model. The list holds at 32,000 until month 12 and rises to 36,902 by month 60.

The first extreme run, with no normal capacity, stopped with a division by zero in `Waiting time` at time 0. The agent guarded the denominator with `MAX(1, Treatments)`. The repeated run reaches 53,535 patients by month 60; with no need after month 12, the list drains to zero and no floor is needed. Its answer lists six changes and three things for the modeller to decide.

The agent's account is a claim. The next cell checks the final spec with our own code, from scratch.

In [ ]:
def numbers(text):
    """Every number written in a text, such as 32,000 or 0.5."""
    found = re.findall(r"(?<![\w.])-?\d{1,3}(?:,\d{3})+(?:\.\d+)?"
                       r"|(?<![\w.])-?\d+(?:\.\d+)?", str(text))
    return {float(x.replace(",", "")) for x in found}


# 1. the final spec, checked by our own code, not by the agent's word
fixed = load(work)
print("loads:", not isinstance(fixed, str), "| units:",
      unit_problems(work) or "balance", "| missing links:",
      link_problems(work, HYPOTHESIS)[0] or "none")
base_run = behaviour(fixed)
print("in balance until month 12:",
      base_run["list at month 0"] == base_run["list at month 12"] == 32000)
for name, value in [("normal_capacity", 0), ("need_increase", -4000)]:
    r = behaviour(fixed, **{name: value})
    print(f"{name} = {value}: runs, floors used:",
          r["stocks held up by a floor"] or "none")
# 2. every number in the answer must come from the task or a tool result
answer = steps[-1]["answer"]
seen = numbers(TASK) | numbers(" ".join(str(s.get("result", ""))
                                        for s in steps))
print("numbers in the answer that no tool returned:",
      sorted(numbers(answer) - seen) or "none")
# 3. units changed while the equation stayed: a person must look at these
before = {e["name"]: e for e in elements(draft3)}
for e in elements(work):
    b = before.get(e["name"])
    if b and b.get("units") != e.get("units"):
        same_eq = equation(b) == equation(e)
        print(f"units of {e['name']}: {b['units']!r} -> {e['units']!r}",
              "(equation unchanged)" if same_eq else "(equation changed)")

Our own checks agree with the agent. The final spec loads, its units balance, no link is missing, the list stays at 32,000 patients until month 12, and both extreme runs finish without a floor at zero. Every number in the answer appears in the task or in a tool result.

The last check prints the two changes a person must look at: declared units that changed while the equation stayed the same. Here both are right, because each equation already said what its variable means: a gap over a smoothing time, and a fraction of capacity. Changing units to fit an equation is sometimes the right repair and sometimes a way to hide an error. The failure gallery shows the second case, and why this check exists.

**Try it 2.**

- *Check.* How many times did the agent call each tool?
- *Apply.* Run the agent's final spec with `normal_capacity=0`. What waiting time does the model report at month 60, and does it mean anything?
- *Extend.* Write a check that the agent's answer names every element whose equation or units it changed.

<details><summary>Worked answer</summary>

*Check.* `load_spec`, `check_units`, `set_equation` and `extreme` three times each, `set_units` and `check_links` twice, `add_constant` and `simulate` once, and one answer: 19 steps.

*Apply.* 53,535 months: the list divided by the guard of one patient a month. It means nothing, and section 3's reviewer says so. The guard keeps the run alive; it does not make the output meaningful, so never report `Waiting time` from a run with no treatments.

*Extend.* Compare each element of the final spec with draft 3. Collect those whose equation or units differ, plus any new element: six names, from `Updating perceived wait` to `Target list`. All six appear in the answer. A check like this catches an agent that changes more than it reports.

</details>

In [ ]:
print(Counter(s.get("tool", "answer") for s in steps))
w = fixed.run(normal_capacity=0)
print("no capacity: Waiting time at month 60 is",
      round(float(w["Waiting time"].iloc[-1])), "months")
changed = [e["name"] for e in elements(work) if e["name"] not in before
           or equation(before[e["name"]]) != equation(e)
           or before[e["name"]].get("units") != e.get("units")]
print("changed:", changed)
print("all named in the answer:", all(n in answer for n in changed))

## 3. A reasoning model reviews the decision rules

**Stage task:** check that each decision rule is one its decision maker could follow. **Kind of AI:** a reasoning model (`reasoning=True`), which thinks before it answers [@wei2022cot]. **Check:** every criticism names a real element and quotes its equation; every suggested molecule is in the catalogue; and code finds the numbers written inside equations on its own. **What stays human:** what each decision maker can see, and which criticisms to act on.

Chapter 7 gave the rule: a decision rule may use only information the decision maker has, which usually means perceived or delayed values rather than true ones [@morecroft1983; @sterman2000]. A second habit is that every number in a rule has a name and a source. Some of this code can check alone, once a person writes down who sees what. The table `VISIBLE` records it from the interviews: GPs see the need presenting to them and their belief about the wait; managers see the list in a monthly report.

In [ ]:
VISIBLE = {   # what each decision maker can see, from the interviews
    "Referrals": ("GPs", {"Need in the population", "Perceived wait"}),
    "Pressure": ("managers", {"Reported list"}),
}


def constants(spec):
    """Names of the variables whose equation is a plain number."""
    out = set()
    for e in spec.get("vars", []):
        try:
            float(e["eq"])
            out.add(lb.name_key(e["name"]))
        except ValueError:
            pass
    return out


def information_problems(spec, visible=VISIBLE):
    """Decision rules that read something their decision maker cannot see.
    Constants are allowed: they are the rule's own parameters."""
    els = {lb.name_key(e["name"]): e for e in elements(spec)}
    fixed_names = constants(spec)
    out = []
    for rule, (who, allowed) in visible.items():
        e = els.get(lb.name_key(rule))
        if e is None:
            continue
        ok = {lb.name_key(a) for a in allowed} | fixed_names
        for r in lb.compile_expr(equation(e)).refs:
            if r not in ok:
                out.append(f"{rule} reads {els[r]['name']}, which is not "
                           f"among what {who} can see")
    return out


def literals(spec):
    """Numbers written inside equations (not 0 or 1), by element."""
    out = []

    def walk(node, name):
        if node[0] == "num" and node[1] not in (0, 1):
            out.append((name, node[1]))
        for child in node[1:]:
            if isinstance(child, tuple):
                walk(child, name)
            elif isinstance(child, list):
                for c in child:
                    walk(c, name)

    fixed_names = constants(spec)
    for e in elements(spec):
        key = lb.name_key(e["name"])
        if e["kind"] != "stock" and key not in fixed_names:
            walk(lb.compile_expr(equation(e)).ast, e["name"])
    return out


print("information:", information_problems(work))
print("numbers inside equations:", literals(work))

In [ ]:
REVIEW = f"""Review the decision rules of this model of a hospital waiting
list. Units are in square brackets.

{listing(work)}

Who decides what, from the interviews: GPs decide Referrals. They see
the patient in front of them and what they believe the wait to be, not
the true wait. Hospital managers decide Overtime, through Pressure.
They see the waiting list in a monthly report.

For each decision rule (Referrals, Pressure, Overtime, Capacity), ask:
does it use only information its decision maker has? Is a number inside
an equation, or a constant, without a stated source? Then suggest
molecules from this catalogue that would formulate a rule better:
{", ".join(lb.molecules())}.

Reply with one JSON object:
{{"criticisms": [{{"element": "...", "quote": "the exact part of its
equation", "kind": "information, constant or other", "why": "...",
"suggestion": "..."}}], "molecules": [{{"key": "...", "where": "...",
"why": "..."}}]}}"""
review = ai.ask_json(REVIEW, system=SYSTEM, reasoning=True, replay=REPLAY)
crit = pd.DataFrame(review["criticisms"])
with pd.option_context("display.max_colwidth", 60, "display.width", 200):
    print(crit[["element", "quote", "kind", "suggestion"]].to_string())

The code finds one information problem: `Pressure` reads `Waiting list`, the true count at this instant, while managers see a monthly report. It also finds three numbers written inside equations: 4,000 and 12 in `Need in the population`, and 0.5 in `Overtime`. Code can find these. It cannot say which matter, or why. That is the reviewer's job.

The reviewer's seven points are specific. It confirms that `Referrals` already uses the GPs' perceived wait. It flags the managers' rule for reading the list without its reporting delay. It flags the unnamed 0.5, the unsourced cap of 0.2, the buried 4,000 and 12, and a target that is a modelling convenience, not the board's target. It also notes that the guard in `Waiting time` gives a meaningless wait when there are no treatments. It suggests three molecules, all after Hines, *Molecules of Structure* [@hines_molecules]. They are Smooth (`smooth`) for the report, Overtime (`overtime`) to replace the free response, and Effect of Fatigue (`effectFatigue`) for the cost of sustained overtime.

Fluent criticism can still point at nothing. The next cell checks that each one quotes a real equation, and that each molecule key exists.

In [ ]:
els = {lb.name_key(e["name"]): e for e in elements(work)}


def points_at_real_equation(c):
    """Does a criticism name a real element and quote its equation?"""
    e = els.get(lb.name_key(c["element"]))
    squash = lambda s: "".join(str(s).split())        # ignore spaces
    return e is not None and squash(c["quote"]) in squash(equation(e))


crit["real"] = [points_at_real_equation(c) for c in review["criticisms"]]
print(crit[["element", "kind", "real"]].to_string())
keys = [m["key"] for m in review["molecules"]]
print("molecules suggested:", keys, "| in the catalogue:",
      [k in lb.molecules() for k in keys])
quoted = " ".join(crit.loc[crit.real, "quote"])
print("numbers inside equations that no criticism mentions:",
      [(n, v) for n, v in literals(work) if v not in numbers(quoted)]
      or "none")

All seven criticisms point at real equations, all three molecules are in the catalogue, and every number inside an equation is covered by a criticism. The checks say the review is about this model; they do not say it is right. Deciding is the modeller's job, and the decisions are written down with the model.

We act on six points and defer one. `Pressure` reads a `Reported list`, a first-order smooth of the list over a reporting delay of 1 month. The 0.5 becomes `Overtime response`, and the 4,000 and 12 become `Base need` and `Month of rise`. The cap of 0.2 and the target stay, with their sources stated. The Overtime molecule would remove the free response, but it needs a desired treatment rate this model does not have, so it waits for a later version.

In [ ]:
DECISIONS = pd.DataFrame([
    ("Pressure", "accept", "add a Reported list: a first-order smooth of "
     "the list over a reporting delay of 1 month (Smooth, smooth)"),
    ("Overtime", "accept", "name the 0.5 Overtime response, an assumption "
     "to test (sensitivity, chapter 12)"),
    ("Maximum overtime", "accept", "keep 0.2, recorded as an assumption; "
     "fatigue (R2) stays in chapter 9"),
    ("Need in the population", "accept", "name Base need and Month of rise"),
    ("Target list", "accept", "keep, with its source stated"),
    ("Waiting time", "accept", "keep the guard; say what it means"),
    ("overtime molecule", "defer", "needs desired treatments; a later "
     "version"),
], columns=["about", "decision", "what we do"])
print(DECISIONS.to_string(index=False))

revised = copy.deepcopy(work)
revised["name"] = "Waiting list: discouraged referrals and working harder"
revised["stocks"].append(stock("rl", "Reported list", "Waiting_list",
                               "patients"))
revised["flows"].append(flow("rep", "Reporting",
                             "(Waiting_list - Reported_list) / "
                             "Reporting_delay", "patients/month", dst="rl"))
revised["vars"] += [var("rd", "Reporting delay", 1, "month"),
                    var("orp", "Overtime response", 0.5, "dimensionless"),
                    var("bn", "Base need", 4000, "patients/month"),
                    var("mr", "Month of rise", 12, "month")]
EDITS = {"Pressure": "Reported_list / Target_list",
         "Overtime": "MIN(Maximum_overtime, Overtime_response * "
                     "MAX(0, Pressure - 1))",
         "Need in the population": "Base_need + STEP(Need_increase, "
                                   "Month_of_rise)"}
for e in revised["vars"]:
    e["eq"] = EDITS.get(e["name"], e["eq"])
case = load(revised)
print(case)
print("units:", unit_problems(revised) or "balance", "| missing links:",
      link_problems(revised, HYPOTHESIS)[0] or "none", "| information:",
      information_problems(revised) or "every rule reads what it can see",
      "| numbers inside equations:", literals(revised) or "none")

In [ ]:
#| fig-alt: "Two panels over 60 months. Left: the waiting list, flat at 32,000 until month 12 and then rising; the agent's spec and the revised spec with a 1-month reporting delay lie on top of each other, and a 6-month delay rises a little higher. Right: capacity rising with overtime after month 12, later with the 6-month delay."
runs = {"agent's spec (no reporting delay)": fixed.run(),
        "revised, reporting delay 1 month": case.run(),
        "revised, reporting delay 6 months": case.run(reporting_delay=6)}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for label, d in runs.items():
    a1.plot(d.index, d["Waiting list"], label=label)
    a2.plot(d.index, d["Capacity"], label=label)
a1.set(title="Waiting list", xlabel="time (months)",
       ylabel="Waiting list (patients)")
a2.set(title="Capacity, with overtime", xlabel="time (months)",
       ylabel="Capacity (patients/month)")
a1.legend()
plt.tight_layout()
plt.show()
for label, d in runs.items():
    print(f"{label}: highest list {d['Waiting list'].max():,.0f} in month "
          f"{d['Waiting list'].idxmax()}, list at month 60 "
          f"{d['Waiting list'].iloc[-1]:,.0f}, highest capacity "
          f"{d['Capacity'].max():,.0f}")

*Figure: the waiting list (left) and capacity (right) over 60 months after need rises by 400 patients a month in month 12. The agent's spec (no reporting delay) and the revised spec (a delay of 1 month) lie on top of each other. A 6-month delay is shown for comparison.*

The revised model passes every check: it loads, its units balance, no link is missing, every rule reads what its decision maker can see, and no number is left inside an equation. The criticism was right in principle and small in effect. With a reporting delay of 1 month the list ends at 36,912 against 36,902 with none; with 6 months it peaks at 37,012 in month 45. Need rises once and stays, so a lag of a month barely matters here. A faster-moving signal would make it matter more, and @sec-testing tests such assumptions. Getting the rule right cost one stock.

**Try it 3.**

- *Check.* How many of the reviewer's criticisms are about information, how many about constants, and how many about something else?
- *Apply.* Make `Pressure` read `Need_in_the_population / Normal_capacity`. Run `information_problems` with `VISIBLE`, then with a looser table in which managers also see the need. Which checks still catch the change?
- *Extend.* Run the revised model with reporting delays of 1, 3, 6, 12 and 24 months. How long must the delay be before the list at month 60 is 1,000 patients higher than with no delay?

<details><summary>Worked answer</summary>

*Check.* Four about constants, two about information (one of them a confirmation that `Referrals` is right) and one other, the guard in `Waiting time`.

*Apply.* With `VISIBLE` the audit flags it: `Pressure` reads `Need in the population`, which managers cannot see. With the looser table it passes, so the information check is only as good as the visibility table a person writes. The link check still catches it: Waiting list to Pressure is missing, because pressure from need opens loop B3.

*Extend.* The list at month 60 is 10, 30, 64, 259 and 1,013 patients higher than with no delay. Only a delay of about two years adds 1,000 patients, and then the list peaks at 38,361. The managers' loop is weak here because overtime can add at most a fifth of capacity, and slowly.

</details>

In [ ]:
print(Counter(crit.kind))
needy = copy.deepcopy(revised)
for e in needy["vars"]:
    if e["name"] == "Pressure":
        e["eq"] = "Need_in_the_population / Normal_capacity"
loose = {**VISIBLE, "Pressure": ("managers", {"Reported list",
                                              "Need in the population"})}
print("strict table:", information_problems(needy))
print("loose table:", information_problems(needy, loose) or "passes")
print("links missing:", link_problems(needy, HYPOTHESIS)[0])
no_delay_end = fixed.run()["Waiting list"].iloc[-1]
for rd in (1, 3, 6, 12, 24):
    end = case.run(reporting_delay=rd)["Waiting list"]
    print(f"reporting delay {rd:>2}: highest list {end.max():,.0f}, "
          f"at month 60 {end.iloc[-1] - no_delay_end:+,.0f} against no "
          "delay")

## 4. Machine learning: discovering structure from data

**Stage task:** find an equation for a rate from data on a stock. **Kind of AI:** machine learning (sparse regression), not a language model. **Check:** we test the method where we know the answer. **What stays human:** the library of candidate terms, the threshold, and which mechanism the equation stands for.

Suppose you have a time series of a stock and suspect it obeys some simple rate equation, but you do not know which. **Sparse regression** turns the search into least squares. Estimate the rate of change $\dot x$ from the data. Build a **library** of candidate terms: a constant, $x$, $x^2$, $x^3$, and so on. Then find the fewest terms that explain $\dot x$:

$$\dot x \approx \Theta(x)\,\xi,\qquad \Theta(x) = [\,1,\ x,\ x^2,\ x^3\,],$$

keeping only the coefficients in $\xi$ that are clearly not zero. This is the idea of SINDy, sparse identification of nonlinear dynamics [@brunton2016]. The simplest version repeats two steps: fit by least squares, then set small coefficients to zero and refit with the rest.

We test it where we know the answer. The Constrained Growth molecule (`logistic`) of @sec-loops is after Hines, *Molecules of Structure* [@hines_molecules]. It has

$$\frac{dP}{dt} = r P - \frac{r}{K}P^2,$$

with $r$ = 0.18 per year and $K$ = 200 (its defaults). We sample it every half year for 40 years, add 1% measurement noise, and ask the regression to find the equation.

In [ ]:
lg = lb.molecule("logistic")
t_obs = np.arange(0, 40.01, 0.5)                       # years
clean = lg.run(T=40).loc[t_obs, "Population"].to_numpy()


def derivative(x, t, width=5):
    # smooth with a moving average, then take central differences
    # np.convolve with weights of 1/width each is a moving average of width
    # points; mode="same" keeps the length. np.gradient(xs, t) is the slope at
    # each point from its two neighbours
    xs = np.convolve(x, np.ones(width) / width, mode="same")
    edge = width // 2
    xs[:edge], xs[-edge:] = x[:edge], x[-edge:]
    return xs[edge:-edge], np.gradient(xs, t)[edge:-edge]


def sindy(x, dx, names, degree=3, threshold=0.1, rounds=10):
    # the library: columns 1, x, x^2, x^3
    lib = np.column_stack([x ** k for k in range(degree + 1)])
    scale = np.linalg.norm(lib, axis=0)                 # comparable columns
    # (norm: the length of each column, the square root of its sum of squares)
    a = lib / scale
    xi = np.linalg.lstsq(a, dx, rcond=None)[0]
    dense = xi / scale
    for _ in range(rounds):
        small = np.abs(xi) < threshold * np.abs(xi).max()
        xi[small] = 0.0
        # ~ flips a mask: refit with the columns that are not small
        xi[~small] = np.linalg.lstsq(a[:, ~small], dx, rcond=None)[0]
    return pd.DataFrame({"least squares": dense, "sparse": xi / scale},
                        index=names)


names = ["1", "P", "P^2", "P^3"]
noisy = clean * (1 + rng.normal(0, 0.01, len(clean)))
x_s, dx = derivative(noisy, t_obs)
fit = sindy(x_s, dx, names)
# " .6f": six decimals with room for a sign
print(fit.to_string(float_format=lambda v: f"{v: .6f}"))
b1, b2 = fit.loc["P", "sparse"], fit.loc["P^2", "sparse"]
print(f"\nRecovered r = {b1:.4f} per year (true 0.18), "
      f"K = {-b1 / b2:.1f} (true 200)")

Plain least squares puts some weight on every term, including a constant of about -0.074 and a cubic term too small to print. After thresholding only $P$ and $P^2$ remain, with coefficients 0.1775 and -0.000879. They give $r$ = 0.1775 per year and $K$ = 201.8, against the true 0.18 and 200.

In [ ]:
#| fig-alt: "Rate of change of the population against the population: dots estimated from noisy data form a parabola through zero, with the true and recovered equations drawn on top of each other."
fig, ax = plt.subplots()
grid = np.linspace(0, 210, 200)
ax.plot(x_s, dx, "o", ms=4, label="estimated from noisy data")
ax.plot(grid, 0.18 * grid - 0.18 / 200 * grid ** 2, label="true equation")
ax.plot(grid, b1 * grid + b2 * grid ** 2, label="recovered equation")
ax.set(title="Rate of change against the stock: logistic growth",
       xlabel="Population (individuals)",
       ylabel="dP/dt (individuals/year)")
ax.legend()
plt.show()

*Figure: the rate of change of the population against the population. Dots are estimated from the noisy series. The recovered equation lies almost on top of the true one.*

Plotting the rate against the stock, a **phase plot**, is the most useful picture in structure discovery: a parabola through zero says "growth that limits itself" before any regression is run.

Two warnings, as in any formulation aid. First, the method finds an equation, not a mechanism: $rP - (r/K)P^2$ is equally the logistic of chapter 5 and, for instance, a diffusion model whose potential adopters are $K-P$. The modeller still decides which story is right. Second, derivatives of noisy data are noisy, and noise is where the method breaks, as the exercise shows.

Why not run it on the waiting list? The data have one stock and no flows, so the regression could fit the net flow but never split it into referrals and treatments. That split is exactly what formulation adds. A sparse regression and a language model are alike in one way: each proposes an equation, and each proposal is a hypothesis to check.

**Try it 4.**

- *Check.* Without the sparse step (the "least squares" column), how many terms have a coefficient that is not zero?
- *Apply.* Raise the noise to 3%. Do you still recover the right terms? How far off are $r$ and $K$?
- *Extend.* Apply the method to the Decay molecule (`decay`) from chapter 5 and confirm it finds a single linear term. Compare the coefficient with the molecule's own parameter.

<details><summary>Worked answer</summary>

*Check.* All four: least squares never sets a coefficient exactly to zero, which is why the sparse step is needed.

*Apply.* With 3% noise and this seed the same two terms are found, with $r$ = 0.1741 per year and $K$ = 205.2. Try other seeds: as noise grows, spurious terms such as $P^3$ start to survive the threshold, and the smoothing width and the threshold become choices you must report.

*Extend.* The Decay molecule's equation is `Decay = Stock / Decay_time` with a decay time of 5 years, so $\dot x = -0.2x$. The method keeps only the linear term, with a coefficient of -0.2026.

</details>

In [ ]:
print("Non-zero least-squares terms:",
      int((fit["least squares"].abs() > 1e-12).sum()))
noisy3 = clean * (1 + np.random.default_rng(SEED).normal(0, 0.03, len(clean)))
f3 = sindy(*derivative(noisy3, t_obs), names)   # * spreads the two results
print(f3["sparse"].to_string(float_format=lambda v: f"{v: .6f}"))
print(f"3% noise: r = {f3.loc['P', 'sparse']:.4f}, "
      f"K = {-f3.loc['P', 'sparse'] / f3.loc['P^2', 'sparse']:.1f}")
dec = lb.molecule("decay")
print(dec.equations())
first = dec.stocks[0]["name"]                   # the name of its first stock
dd = dec.run(T=40).loc[t_obs, first].to_numpy()
fd = sindy(*derivative(dd * (1 + np.random.default_rng(SEED).normal(
    0, 0.01, len(dd))), t_obs), ["1", "x", "x^2", "x^3"])
print(fd["sparse"].to_string(float_format=lambda v: f"{v: .6f}"))
print("Decay parameters:", dec.params)

## 5. Documentation: a table by code, a description by a model

**Stage task:** document the model so others can read, check and reuse it [@martinezmoyano2012]. **Kind of AI:** generative. **Check:** every variable the description names exists, and every number it states appears in the spec. **What stays human:** the source of every constant, and the final text.

Split the work by what each does well. Code writes the equations and units table, because it can copy them without error. The modeller writes the sources, because only the modeller knows them. A language model writes the plain-language paragraph, because that is what it is good at. First, a check that every constant has a source.

In [ ]:
SOURCES = {
    "Base need": "chapter 7 case: 4,000 patients/month",
    "Need increase": "scenario: a 10% rise, as in chapters 7 and 8",
    "Month of rise": "scenario",
    "Normal capacity": "set equal to need, so the list starts in balance",
    "Minimum treatment time": "chapter 1",
    "Reference wait": "starting wait: 32,000 / 4,000 (Little's law, ch. 6)",
    "Perception time": "assumed in chapter 6",
    "Referral sensitivity": "assumed in chapter 6",
    "Target list": "the starting list, so the model starts in balance",
    "Maximum overtime": "assumed; the consultant: 'only for so long'",
    "Overtime response": "assumed; to test in chapter 12",
    "Reporting delay": "the list is reported monthly",
}
missing_sources = sorted(case.params.keys() - {lb.name_key(n)
                                               for n in SOURCES})
print("constants with no source:", missing_sources or "none")


def doc_table(spec, sources):
    """A Markdown table of every element: equation, units and source."""
    rows = ["| Name | Kind | Equation | Units | Source |",
            "|---|---|---|---|---|"]
    for e in elements(spec):
        rows.append(f"| {e['name']} | {e['kind']} | `{equation(e)}` | "
                    f"{e.get('units', '')} | {sources.get(e['name'], '')} |")
    return "\n".join(rows)


TABLE = doc_table(revised, SOURCES)
print(TABLE)

In [ ]:
DOC = f"""Write a plain-language description of this model for a hospital
manager: one paragraph of five to seven sentences. Put every model
variable you name in backticks, written exactly as in the table. Use
only numbers that appear in the table.

{TABLE}"""
description = ai.ask(DOC, system=SYSTEM, replay=REPLAY)
print(description)

In [ ]:
known = {lb.name_key(e["name"]) for e in elements(revised)}
named = re.findall(r"`([^`]+)`", description)
print("names in backticks:", len(named), "| not in the model:",
      [n for n in named if lb.name_key(n) not in known] or "none")
in_spec = numbers(" ".join(equation(e) for e in elements(revised))) | \
    {revised["T"], revised["dt"]}
print("numbers in the description that are not in the spec:",
      sorted(numbers(description) - in_spec) or "none")

The description is clear and names 14 variables, all in backticks and all real. One number is not in the spec: 4,400. It is right, 4,000 plus 400, but the model computed it, and the contract says numbers come from tools. The check does not decide; it puts the number in front of a person. We verified it from a run (Try it 5) and kept it.

The check also guards against drift. A description written once goes stale when the model changes; rerun the check after every change and it flags the numbers that no longer hold.

**Try it 5.**

- *Check.* Verify the flagged number from a run of the revised model.
- *Apply.* Change `Perception time` to 4 months in a copy of the spec and rerun the number check on the old description. What does it flag?
- *Extend.* Which stocks and which constants does the description leave out? Is leaving them out a fault?

<details><summary>Worked answer</summary>

*Check.* `Need in the population` is 4,000 patients a month at month 0 and 4,400 at month 60, so the sentence is right.

*Apply.* It flags 6 as well as 4,400: the description still says the perception time is 6 months. The check finds the stale sentence at once.

*Extend.* All three stocks are named. Seven constants are not named: `Base need`, `Minimum treatment time`, `Month of rise`, `Need increase`, `Overtime response`, `Reference wait` and `Referral sensitivity`. Some, such as the base need, appear as numbers only. For a manager's paragraph that is fine; the table holds them all. For a technical appendix you would ask for every constant and add that to the check.

</details>

In [ ]:
need = case.run()["Need in the population"]
print("need at month 0 and 60:", need.iloc[0], need.iloc[-1])
stale = copy.deepcopy(revised)
for e in stale["vars"]:
    if e["name"] == "Perception time":
        e["eq"] = "4"
in_stale = numbers(" ".join(equation(e) for e in elements(stale))) | \
    {stale["T"], stale["dt"]}
print("after the change:", sorted(numbers(description) - in_stale))
named_keys = {lb.name_key(n) for n in named}
print("stocks not named:", [s["name"] for s in revised["stocks"]
                            if lb.name_key(s["name"]) not in named_keys])
print("constants not named:", sorted(k for k in case.params
                                     if k not in named_keys))

## Failure gallery

Five ways AI goes wrong in formulation. Each is shown below, by a replayed reply or by code, and each is caught by a check. Two of the replies here hold faults we planted for teaching: the delay equation of the first item and the agent run of the last.

**1. A hallucinated function.** Asked for a third-order delay, a model reaches for Vensim's `DELAY3`.

In [ ]:
ASK_DELAY = ("In the loopbricks equation language, write the equation for "
             "Referrals arriving, the outflow of a third-order delay of 1.5 "
             "months on Referrals written. Reply with the equation only.")
delay_eq = ai.ask(ASK_DELAY, system=SYSTEM, replay=REPLAY).strip()
print(delay_eq)
try:
    lb.compile_expr(delay_eq)
except lb.LoopBricksError as e:
    print("caught:", e)

The compiler catches it at once. In our language a delay is a chain of stocks: the Material Delay molecule (`material`) of chapter 6, after Hines, *Molecules of Structure* [@hines_molecules]. It has three stocks, each draining over a third of the delay time. Section 1 met the same failure with `SMOOTH`.

**2. Dimensionally inconsistent equations.** Three classic slips in `Treatments`: adding the list to a rate, a gap with no time to close it, and multiplying by a time instead of dividing. All three load.

In [ ]:
SLIPS = {"a stock added to a flow": "Capacity + Waiting_list",
         "a gap with no time to close it": "Waiting_list - Target_list",
         "multiplied by a time, not divided": "Capacity * "
                                              "Minimum_treatment_time"}
for label, eq in SLIPS.items():
    slip = copy.deepcopy(revised)
    for f in slip["flows"]:
        if f["name"] == "Treatments":
            f["eq"] = eq
    print(f"{label}: loads: {not isinstance(load(slip), str)}; "
          f"{unit_problems(slip)}")

Each one loads and runs, and each is caught by the units checker. Loading cannot see units at all, which is why a spec that loads is not yet a model.

**3. A decision rule that reads the true value.** Let the GPs' referral rule read the true `Waiting_time` instead of their perceived wait. The model still loads and its units still balance: the true wait and the perceived wait are both in months.

In [ ]:
true_wait = copy.deepcopy(revised)
for f in true_wait["flows"]:
    if f["name"] == "Referrals":
        f["eq"] = f["eq"].replace("Perceived_wait", "Waiting_time")
print("Referrals on the true wait:", information_problems(true_wait))
print("draft 3:", information_problems(draft3))
print("loads:", not isinstance(load(true_wait), str), "| units:",
      unit_problems(true_wait) or "balance")

Only the information audit catches it, because the error is in what the rule knows, not in its form. The same audit catches draft 3's fault from section 1, where the managers' pressure read the GPs' perceived wait. The audit is only as good as the visibility table a person writes, as Try it 3 showed.

**4. Magic constants.** A **magic constant** is a number written inside an equation, with no name and no source. Draft 3 had three; the revised model has none, and every named constant has a source.

In [ ]:
print("draft 3, numbers inside equations:", literals(draft3))
print("revised, numbers inside equations:", literals(revised) or "none")
print("revised, constants with no source:", missing_sources or "none")

**5. An agent that changes the units, not the equation.** This last spec has a planted fault: `Pressure` is `Reported_list - Target_list`, a difference in patients, where a ratio was meant. The task tells a hasty agent only to make `check_units` pass. The four replies of this run were written to show the failure; they are not a live model's output.

In [ ]:
lenient = copy.deepcopy(revised)
for e in lenient["vars"]:
    if e["name"] == "Pressure":
        e["eq"] = "Reported_list - Target_list"     # forgot to divide
    if e["name"] == "Overtime":
        e["eq"] = "Maximum_overtime * TABLE(Pressure, 1, 0, 2, 1)"
cheat_tools, cheat_work = make_tools(lenient)
CHEAT_TASK = ("check_units fails on this spec. Make it pass.\n\n"
              + listing(lenient))
cheat = ai.agent(CHEAT_TASK, cheat_tools, replay=REPLAY, max_steps=6,
                 system="You fix model specs with tools, quickly.")
ai.show(cheat, width=110)

In [ ]:
def table_ranges(spec, run):
    """For each TABLE, the share of the run its input spends outside the
    lookup's points, where the lookup is flat."""
    out = {}
    names = {lb.name_key(e["name"]): e["name"] for e in elements(spec)}
    for e in elements(spec):
        tree = lb.compile_expr(equation(e)).ast
        stack = [tree]
        while stack:
            n = stack.pop()
            if n[0] == "call" and n[1] == "TABLE" and n[2][0][0] == "name":
                xs = [p[1] for p in n[2][1::2]]
                x = run[names[n[2][0][1]]]
                out[e["name"]] = float(((x < min(xs)) | (x > max(xs)))
                                       .mean())
            for part in n[1:]:
                if isinstance(part, tuple):
                    stack.append(part)
                elif isinstance(part, list):
                    stack += part
    return out


print("units:", unit_problems(cheat_work) or "balance")
for e in elements(cheat_work):
    old = next(x for x in elements(lenient) if x["name"] == e["name"])
    if old["units"] != e["units"] and equation(old) == equation(e):
        print(f"flag: units of {e['name']} changed from {old['units']!r} "
              f"to {e['units']!r}, equation unchanged")
gamed = load(cheat_work)
print("share of the run outside the lookup's points:",
      table_ranges(cheat_work, gamed.run()))
print("highest capacity:", round(gamed.run()["Capacity"].max()),
      "against", round(case.run()["Capacity"].max()), "in the revised model")

The agent did what it was asked: `check_units` passes. It declared `Pressure` to be in patients, and the checker cannot object, because a `TABLE` accepts any input. But the lookup's points run from 1 to 2, and a pressure in patients lies outside them for 99.6% of the run. The lookup sits on its flat end, so overtime is either off or at its cap of a fifth. Capacity jumps to 4,800 patients a month, against 4,307 in the revised model.

Two checks catch it. The units diff flags a declared unit that changed while the equation did not, the check of section 2. The table range check finds an input that lives outside its lookup. An agent told to make a check pass will find the cheapest way to make it pass. Give it the goal, not the check, and check the result yourself.

**Try it 6.**

- *Check.* Which two checks caught the agent of item 5, and which check did it satisfy?
- *Apply.* Write a stricter units checker that also requires every `TABLE` input to be dimensionless. Does the cheating agent's spec pass it? Does the revised model?
- *Extend.* Write a test that rejects any change of declared units with the equation unchanged, unless a person has approved that element by name. Run it on both agent runs.

<details><summary>Worked answer</summary>

*Check.* The units diff and the table range check caught it. It satisfied `check_units`, which it was told to make pass.

*Apply.* The strict checker reports that `Overtime` has a `TABLE` input in patients, so the cheat fails; the revised model still balances. Strict checks are better, but an agent can still game a strict check in some other way. The defence is several independent checks and a person who reads the diff.

*Extend.* With the two changes of section 2 approved, the section 2 agent has no unapproved change. The gallery agent has one, `Pressure`. The approved set is the person's record of what they read and accepted.

</details>

In [ ]:
def strict_unit_problems(spec):
    """unit_problems, plus: the input of every TABLE must be dimensionless."""
    out = unit_problems(spec)
    units = {lb.name_key(e["name"]): parse_units(e["units"])
             for e in elements(spec)}
    units["TIME"] = parse_units(spec.get("timeUnit", ""))
    for e in elements(spec):
        stack = [lb.compile_expr(equation(e)).ast]
        while stack:
            n = stack.pop()
            if n[0] == "call" and n[1] == "TABLE":
                u = infer(n[2][0], units, [], e["name"])
                if u:
                    out.append(f"{e['name']}: a TABLE input in "
                               f"{show_units(u)}")
            for part in n[1:]:
                if isinstance(part, tuple):
                    stack.append(part)
                elif isinstance(part, list):
                    stack += part
    return out


print("strict, cheat's spec:", strict_unit_problems(cheat_work))
print("strict, revised:", strict_unit_problems(revised) or "balance")


def unapproved_units(old, new, approved):
    """Units changed with the equation unchanged, not signed off by a
    person."""
    before = {e["name"]: e for e in elements(old)}
    return [e["name"] for e in elements(new) if e["name"] in before
            and before[e["name"]]["units"] != e["units"]
            and equation(before[e["name"]]) == equation(e)
            and e["name"] not in approved]


APPROVED = {"Updating perceived wait", "Overtime"}   # reviewed in section 2
print("section 2 agent:", unapproved_units(draft3, work, APPROVED) or "none")
print("gallery agent:", unapproved_units(lenient, cheat_work, APPROVED))

## Check it

Whoever wrote the equations, a person, a language model or an agent, run the same five checks before a model leaves formulation. The cell runs them on the revised model.

1. **It loads.** Names, syntax, flow ends and no simultaneous equations.
2. **Units balance.** Every term of a sum in the same units; every flow in its stock's units per unit of time.
3. **It is the hypothesis, and its rules are honest.** Every link of the hypothesis is in the equations, and every decision rule reads only what its decision maker can see.
4. **It behaves.** It starts in balance, and extreme runs finish without a floor holding a stock up.
5. **The time step is small enough.** Halving `dt` barely moves the results.

In [ ]:
final = load(revised)
print("1. loads:", not isinstance(final, str))
print("2. units:", unit_problems(revised) or "balance")
print("3. links missing:", link_problems(revised, HYPOTHESIS)[0] or "none",
      "| information:", information_problems(revised) or "clean")
d = final.run()
print("4. in balance before the rise:",
      bool(np.allclose(d.loc[:12, "Waiting list"], 32000)))
for name, value in [("normal_capacity", 0), ("need_increase", -4000)]:
    r = behaviour(final, **{name: value})
    print(f"   {name} = {value}: lowest stocks "
          f"{r['lowest value of each stock']}, floors used: "
          f"{r['stocks held up by a floor'] or 'none'}")
half = final.run(dt=0.0625)["Waiting list"].reindex(d.index)
print("5. halving dt moves the list by at most",
      round(float((half - d["Waiting list"]).abs().max()), 1), "patients")

All five pass. Before the rise in need the list stays at 32,000 patients. Both extreme runs finish with every stock at or above zero and no floor used. Halving `dt` moves the list by at most 5.6 patients.

## Running case

This chapter's increment is the formulation of two loops of the waiting-list hypothesis, drafted with AI help and checked by code at every step:

| Element | What it is | Value |
|---|---|---|
| `Perceived wait` (stock) | the GPs' belief about the wait, a first-order smooth | perception time 6 months (assumed in chapter 6) |
| `Referrals` | need times the perceived wait relative to the reference wait, to the power of minus the sensitivity | reference wait 8 months, sensitivity 0.3 |
| `Reported list` (stock) | what managers see: the list smoothed over the reporting delay | reporting delay 1 month |
| `Pressure` | reported list over the target list | target 32,000 patients, the starting list |
| `Overtime` | response times the excess pressure, capped | response 0.5, cap 0.2 (both assumed) |
| `Capacity` | normal capacity scaled up by overtime | normal capacity 4,000 patients/month |

Each AI output has a record: what was asked, its replay key, the check it met, the result and what a person decided. The record is part of the model, as a data source is.

In [ ]:
def replay_file():
    """The replay records of this chapter, from the local file or GitHub."""
    if os.path.exists(REPLAY):
        return json.load(open(REPLAY))
    with urlopen(REPLAY) as r:
        return json.load(r)


def key(messages, system=SYSTEM, reasoning=False):
    return ai.request_key(messages, system=system, reasoning=reasoning)


user = lambda text: [{"role": "user", "content": text}]
RECORD = pd.DataFrame([
    ("1", "generative", "draft spec", key(user(PROMPT)),
     "loads", "SMOOTH is not a function", "fix asked"),
    ("1", "generative", "fixed spec", key(fix_request(reply1, load(draft1))),
     "loads", "variables in a circle", "fix asked"),
    ("1", "generative", "fixed spec", key(messages3),
     "loads", "loads", "accepted as a draft"),
    ("2", "agent", f"{len(steps)} steps", "one per step",
     "units, links, runs", "all pass", "units changes reviewed"),
    ("3", "reasoning", "review", key(user(REVIEW), reasoning=True),
     "quotes, keys", f"{int(crit.real.sum())} of {len(crit)} real",
     "6 acted on, 1 deferred"),
    ("5", "generative", "description", key(user(DOC)),
     "names, numbers", "one number to verify", "verified, kept"),
], columns=["section", "kind", "output", "replay key", "check", "result",
            "decision"])
records = replay_file()
RECORD["how"] = [records[k]["how"] if k in records else "(agent steps)"
                 for k in RECORD["replay key"]]
print(RECORD.drop(columns="how").to_string(index=False))
print("\nhow they were made:", Counter(r["how"] for r in records.values()))

The record shows the whole formulation. Three generative drafts led to a spec that loads, and an agent repaired it in 19 steps. A reasoning model's review led to six changes and one deferral. A description was checked and one number verified. Every record says how it was made, and here all were written for the book. @sec-learning starts Stage 3, where this model meets the data.

## Tell the decision maker

"We used AI to draft and review the equations for two of the loops behind the waiting list. Every draft was checked by code: it had to load, balance its units, contain the causes the staff described, and behave sensibly in extreme cases. The AI found real problems, such as a number nobody had given it; it also made them, and the checks caught those. Three numbers in the model are assumptions, the size and speed of overtime and how strongly GPs react to waits, and the next chapters test how much they matter. Ask for the record: for every AI output it says what was asked and who decided."

## Going further

- **Units libraries.** Our checker is short on purpose. The `pint` package handles unit conversions and many more cases; Vensim and Stella check units too, and @sec-vensim-stella shows how to bring their models into Python.
- **Polarity checks.** The link checker finds links, not their signs. A stronger check perturbs each cause in a short run and confirms that the effect moves in the direction the edge list says.
- **Structured tool calls.** Live agents can use the API's own tool-use format instead of our one-JSON-object protocol; the tools and the checks stay the same.
- **Structure discovery at scale.** The `pysindy` package implements SINDy [@brunton2016] with richer libraries, derivative estimators and sparse solvers. Try it on a two-stock molecule such as Diffusion.
- **Language models inside models.** A different use of the same technology puts language models in the model, as simulated people; see @ghaffarzadegan2024. That is a modelling choice to test, not a formulation aid.

## Mini-project

Formulate a third loop of chapter 3's hypothesis, R2 Burnout: overtime builds fatigue, fatigue drives staff away, and fewer staff mean less capacity. Use the whole pipeline of this chapter. Ask a model for a spec from the links, load it and feed the errors back. Give an agent the tools of section 2, with the new links in the hypothesis. Have a reasoning model review the new rules and check its review. Write the documentation and check it. Then compare your formulation with the Effect of Fatigue molecule and with @sec-patterns, and finish with a record table like the one above.

## Further reading

On the formulation of decision rules, see @morecroft1983 and the formulation chapters of @sterman2000; @richardson1981 remains a clear guide to writing equations. For good practice in modelling, including documentation, see @martinezmoyano2013 and @martinezmoyano2012. On language models in system dynamics, @akhavan2024 says how to use them and how not to, and @hosseinichimeh2024 builds causal diagrams from text. On agents that reason and act with tools, see @yao2022react, and on why fluent models invent things, @ji2023hallucination. For structure discovery, start with @brunton2016.